# 02 - load Firnberg DMS, seal wet-lab holdout, build splits

Standalone-runnable. Reads `data/raw/BLAT_ECOLX_Firnberg_2014.csv`, seals the 13 wet-lab validation variants (fold=-1), builds random, modulo, and contiguous 5-fold splits (seed 42), writes `data/processed/tem1_firnberg_processed.parquet` and `split_definitions.json`.

The wet-lab panel's identity came from translating the mutagenesis primers (see `data/interim/wetlab_panel_reconciliation.csv`).

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import pandas as pd, numpy as np, json
SEED=42; np.random.seed(SEED)
df = pd.read_csv(RAW/'BLAT_ECOLX_Firnberg_2014.csv')
assert list(df.columns)[:4]==['mutant','mutated_sequence','DMS_score','DMS_score_bin']
assert len(df)==4783, len(df)
print('Firnberg rows:', len(df), '| balance:', round(df.DMS_score_bin.mean(),4))

### Parse mutant strings, reconstruct WT, verify positions

In [ ]:
import re
pat=re.compile(r'^([A-Z])(\d+)([A-Z])$')
df['wt_aa']=df.mutant.str.extract(r'^([A-Z])')
df['position_linear']=df.mutant.str.extract(r'(\d+)').astype(int)
df['mut_aa']=df.mutant.str.extract(r'([A-Z])$')
print('linear range', df.position_linear.min(),'-',df.position_linear.max(),
      '| positions', df.position_linear.nunique())

### Seal the 13 wet-lab validation variants (never trained, tuned, or selected on)

In [ ]:
SEALED=['F58N','S80N','E87F','K109Q','W163S','E166A','N168A',  # 7 positive controls
        'S68A','I93S','I93Y','E164N','T187K','V258G']         # 6 negative controls
df['excluded_wetlab_validation']=df.mutant.isin(SEALED)
assert df.excluded_wetlab_validation.sum()==13, df.excluded_wetlab_validation.sum()
df['position_ambler']=df.position_linear+2  # +2 offset consistent with anchors S68A/S70,
                                            # Lys73A/K71, E164N/E166; standard TEM-1 mature
                                            # numbering. Reporting-only; not used for splits/features.
pool=df[~df.excluded_wetlab_validation]
print('sealed', df.excluded_wetlab_validation.sum(), '| trainable pool', len(pool),
      '| pool balance', round(pool.DMS_score_bin.mean(),4))

### Build the three splits (random, modulo, contiguous), seed 42
Sealed variants get fold=-1 in all three. Contiguous is the honest headline.

In [ ]:
from sklearn.model_selection import KFold
pool_idx=pool.index.values; positions=np.sort(pool.position_linear.unique())
df['fold_random']=-1; df['fold_modulo']=-1; df['fold_contiguous']=-1
# random: variant-level KFold (positions LEAK across folds -> optimistic)
kf=KFold(5,shuffle=True,random_state=SEED)
for k,(_,te) in enumerate(kf.split(pool_idx)): df.loc[pool_idx[te],'fold_random']=k
# modulo: position %% 5 (each position wholly in one fold)
df.loc[pool.index,'fold_modulo']=(pool.position_linear%5).values
# contiguous: 5 contiguous position blocks (whole regions held out)
edges=np.array_split(positions,5); pos2blk={p:b for b,blk in enumerate(edges) for p in blk}
df.loc[pool.index,'fold_contiguous']=pool.position_linear.map(pos2blk).values
for c in ['fold_random','fold_modulo','fold_contiguous']:
    print(c, 'fold sizes', df[df[c]>=0][c].value_counts().sort_index().tolist())

In [ ]:
assert (df.loc[df.excluded_wetlab_validation,['fold_random','fold_modulo','fold_contiguous']]==-1).all().all()
df.to_parquet(PROCESSED/'tem1_firnberg_processed.parquet')
blocks=[[int(b[0]),int(b[-1])] for b in edges]
json.dump({'seed':SEED,'contiguous_blocks_linear':blocks,'sealed':SEALED},
          open(PROCESSED/'split_definitions.json','w'), indent=2)
print('wrote processed parquet + split_definitions.json'); print('contiguous blocks:', blocks)